# Period effects and the Mundlak term

A shock that hits every SKU in one period (a holiday, weather, a competitor campaign) moves demand and, often, price. A smooth trend cannot absorb it, so the elasticity is biased toward zero.

This notebook fits the same confounded panel three ways: no period effect, `time_effects="period"`, and period effects plus `mundlak=True`. MCMC settings are small; increase them for a real fit.


In [1]:
import numpy as np
import pymc as pm

from pypricing import LogLogDemandModel, PanelColumns, generate_mock_data

df, truth = generate_mock_data(
    n_periods=16,
    n_skus=8,
    hierarchy_levels=(2,),
    n_controls=0,
    include_seasonality=False,
    price_shock_sigma=0.2,
    noise_sigma=0.1,
    common_shock_sigma=0.4,
    common_endogeneity=1.2,
    mundlak_coef=1.5,
    true_elasticity=-1.0,
    random_state=0,
    return_truth=True,
)
panel = PanelColumns(group_columns=("category_1",), quantity_floor=1e-12)
config = {"mu_elasticity": {"dist": pm.Normal, "kwargs": {"mu": -1.0, "sigma": 1.0}}}
fit_kwargs = dict(draws=200, tune=200, chains=2, random_seed=0, progressbar=False)


In [2]:
specs = {
    "none": {},
    "period": {"time_effects": "period"},
    "period + mundlak": {"time_effects": "period", "mundlak": True},
}
label_to_i = {lab: i for i, lab in enumerate(truth.sku_labels)}
rows = []
for name, spec in specs.items():
    model = LogLogDemandModel(panel_columns=panel, model_config=config, **spec)
    model.fit(df, **fit_kwargs)
    order = [label_to_i[str(s)] for s in model.sku_levels_]
    true_e = truth.elasticity_sku[order]
    mean_e = model.idata.posterior["elasticity_sku"].mean(("chain", "draw")).values
    rows.append((name, float(np.mean(mean_e - true_e))))

for name, bias in rows:
    print(f"{name:20s}  mean elasticity − truth = {bias:+.3f}")


Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [mu_alpha, sigma_alpha_group_0, eta_alpha_group_0, sigma_alpha_sku, eta_alpha_sku, mu_elasticity, sigma_elasticity_group_0, eta_elasticity_group_0, sigma_elasticity_sku, eta_elasticity_sku, sigma]
Sampling 2 chains for 200 tune and 200 draw iterations (400 + 400 draws total) took 3 seconds.
There were 47 divergences after tuning. Increase `target_accept` or reparameterize.
We recommend running at least 4 chains for robust computation of convergence diagnostics
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [mu_a

none                  mean elasticity − truth = +0.498
period                mean elasticity − truth = +0.004
period + mundlak      mean elasticity − truth = +0.002


The first fit attributes the common demand shock to price, so the elasticity moves toward zero (the bias is positive). Period effects put that shock in `lambda_period`. The Mundlak term matters when baseline demand is correlated with a SKU's price level and the intercepts are hierarchical: without it, shrinkage of the intercept loads that correlation onto the elasticity too.

Neither replaces instruments. A shock that hits one SKU in one period, and that the price reacts to, is still in the residual.
